# Vehicle CO2 Emissions Estimator — Clean Pipeline

A concise, reproducible walkthrough of the final estimation system. For the full
exploratory journey — EDA, the worst-error diagnostic that uncovered the plug-in
hybrid problem, and every decision along the way — see `01_exploration.ipynb`.

**The core idea:** a vehicle fleet is three different estimation problems, not one.

| Regime | Fuel types | Method |
|---|---|---|
| Combustion (ICE) | petrol, diesel, lpg, e85, ng | ML model (XGBoost) |
| Zero-tailpipe | electric, hydrogen | Assign 0 g/km |
| Plug-in hybrid | petrol/electric, diesel/electric | Certified value + real-world correction |

All reusable logic lives in `src/emissions.py`; this notebook orchestrates it.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")   # so we can import the shared module

import emissions as em
import pandas as pd

## 2. Load data

The raw EEA table holds ~10.8M registration events. We pull **distinct vehicle
specifications** (with electric range for the PHEV branch). De-duplicating before
the split prevents identical vehicles leaking across the train/test boundary.

In [2]:
car_data = em.load_car_data(n=160000, include_zr=True)
print(car_data.shape)
car_data.head()

(153739, 7)


,Ewltp (g/km),Ft,Cr,M (kg),Ec (cm3),Ep (KW),Zr
0,NaN,diesel,M1,NaN,NaN,12.0,NaN
1,NaN,diesel,M1,NaN,1968.0,90.0,NaN
2,NaN,diesel,M1,NaN,1968.0,142.0,NaN
3,NaN,diesel,M1,NaN,1995.0,110.0,NaN
4,NaN,diesel,M1,NaN,1997.0,81.0,NaN


## 3. Split, then clean

Split first, so no decision is made using test data. Then remove a cluster of
combustion rows carrying a placeholder emissions value (~57 g/km, physically
impossible for a fuelled engine) — a systematic reporting gap, not real data.

In [3]:
train_X, val_X, train_y, val_y = em.split_data(car_data, test_size=0.30)

train_X, train_y, _ = em.clean_placeholders(train_X, train_y)
val_X,   val_y,   _ = em.clean_placeholders(val_X,   val_y)

print("train:", train_X.shape, " val:", val_X.shape)

train: (105919, 5)  val: (45377, 5)


## 4. Scope the ML model to combustion vehicles

The model only applies where emissions track the hardware. Plug-in hybrids
(unobservable usage), pure electric / hydrogen (zero, handled by the router), and
unknown fuels are excluded from training. Removing the hybrids is not hiding hard
cases — their emissions depend on driver charging behaviour that no vehicle
dataset can capture, so they are a genuinely different problem.

In [4]:
ice_train_X, ice_train_y = em.filter_ice(train_X, train_y)
ice_val_X,   ice_val_y   = em.filter_ice(val_X,   val_y)

print("ICE train:", ice_train_X.shape, " ICE val:", ice_val_X.shape)
print(ice_train_X["Ft"].value_counts())

ICE train: (67169, 5)  ICE val: (28791, 5)
Ft
petrol    39651
diesel    26593
lpg         855
e85          62
ng            8
Name: count, dtype: int64


## 5. Train the combustion model

A three-model bake-off (decision tree / random forest / XGBoost), each with its
preprocessing tuned by cross-validation. The winner is chosen by **cross-validated
RMSE** — never by the test set, which is reported only as an honest final estimate.

In [5]:
ice_model, results = em.train_ice_model(
    ice_train_X, ice_train_y, ice_val_X, ice_val_y, cv=5
)

for name, r in results.items():
    print(f"{name:14s} | CV RMSE={r['cv_rmse']:7.3f} | "
          f"Test RMSE={r['test_rmse']:7.3f}  MAE={r['test_mae']:7.3f}  R2={r['test_r2']:.4f}")

best = min(results, key=lambda k: results[k]['cv_rmse'])
print(f"\nWinner (by CV): {best} | honest test RMSE {results[best]['test_rmse']:.3f}")

decision_tree  | CV RMSE= 15.145 | Test RMSE= 14.961  MAE=  8.700  R2=0.8964
random_forest  | CV RMSE= 14.285 | Test RMSE= 14.311  MAE=  8.167  R2=0.9052
xgboost        | CV RMSE= 14.043 | Test RMSE= 14.285  MAE=  8.349  R2=0.9055

Winner (by CV): xgboost | honest test RMSE 14.285


### Result

The combustion-only model reaches roughly **MAE 8.3 g/km (~5% of mean emissions),
R² 0.91** on held-out data. Training on combustion vehicles alone (rather than all
fuel types mixed) cut RMSE by ~25% — proportionally more than MAE — confirming that
plug-in hybrids were the source of the large-error tail. (Numbers populate when you
run this against the live dataset.)

## 6. Save the model

Persist the trained pipeline so the Streamlit app can load it without retraining.

In [6]:
em.save_model(ice_model, "../models/ice_model.pkl")
print("saved to ../models/ice_model.pkl")

saved to ../models/ice_model.pkl


## 7. The three-regime router

The router sends each vehicle to the right method. Below we apply it to the full
dataset (all fuel types) and spot-check that each regime behaves correctly.

In [7]:
phev_median = em.phev_median_ewltp(car_data)
print("PHEV median certified Ewltp:", phev_median)

# Estimate CO2 for every distinct spec via the router.
car_data = car_data.copy()
car_data["co2_estimate"] = em.estimate_co2_frame(car_data, ice_model, phev_median)

# Spot-check one vehicle per regime.
for ft in ["petrol", "electric", "hydrogen", "petrol/electric"]:
    sample = car_data[car_data["Ft"] == ft].head(1)
    if len(sample):
        row = sample.iloc[0]
        print(f"{ft:16s} -> {row['co2_estimate']:.1f} g/km")

PHEV median certified Ewltp: 43.0
petrol           -> 146.0 g/km
electric         -> 0.0 g/km
hydrogen         -> 0.0 g/km
petrol/electric  -> 62.4 g/km


## 8. Sanity-check the PHEV correction

There is no real-world ground truth to score PHEVs against, so we check
*plausibility*: corrected values should sit above their certificates, and an
engine-heavy PHEV should land in a realistic combustion-like range rather than the
implausibly low certified figure.

In [8]:
phev = car_data[car_data["Ft"].str.contains("/electric", na=False)].copy()
phev["lift"] = phev["co2_estimate"] / phev["Ewltp (g/km)"]

print(phev[["Ewltp (g/km)", "Zr", "co2_estimate", "lift"]].describe())

# Largest-engine PHEVs: certified vs corrected
big = phev.sort_values("Ec (cm3)", ascending=False).head(5)
big[["Ft", "M (kg)", "Ec (cm3)", "Ep (KW)", "Zr", "Ewltp (g/km)", "co2_estimate"]]

       Ewltp (g/km)            Zr  co2_estimate          lift
count  16868.000000  11856.000000  16988.000000  16868.000000
mean      73.434551     94.858553    100.712147      1.331240
std       67.774012     47.181942     96.586368      0.096522
min        2.000000     11.000000      2.500000      1.250000
25%       21.000000     66.000000     27.500000      1.250000
50%       43.000000     83.000000     58.750000      1.250000
75%      119.000000    112.000000    165.000000      1.450000
max      406.000000    664.000000    686.800000      1.700000


,Ft,M (kg),Ec (cm3),Ep (KW),Zr,Ewltp (g/km),co2_estimate
153394,petrol/electric,1985.0,6498.0,607.0,13.0,350.0,595.0
153687,petrol/electric,1985.0,6498.0,607.0,NaN,406.0,588.7
153682,petrol/electric,1967.0,6498.0,607.0,13.0,404.0,686.8
100755,petrol/electric,1967.0,6498.0,607.0,12.0,153.0,260.1
150007,petrol/electric,1967.0,6498.0,607.0,12.0,275.0,467.5


## Summary

- **Combustion vehicles:** ML model, honest held-out MAE ~8.3 g/km (~5% of mean).
- **Electric / hydrogen:** assigned 0 g/km.
- **Plug-in hybrids:** certified value corrected for real-world usage, scaled by
  electric range.

The key finding is methodological: **one third of the problem should not be
modelled at all** — the right approach is to identify each regime and apply the
appropriate tool. See `WRITEUP.md` for how this maps onto financed-emissions
(PCAF) accounting.